# 记忆检索模式（Memory Retrieval Patterns）

> **记忆系统的好坏，取决于它能否在正确的时机呈现正确的信息。**

想象一位图书管理员处理一个研究问题。他不会只查一个目录。他先查书名索引找精确匹配，再浏览书架找相关主题，还会请教同事寻找不那么明显的关联。每种方法都能接住其他方法漏掉的东西。组合起来才能得到最好的答案。

存储记忆是容易的部分。难的是检索：在智能体需要时找到正确的记忆。朴素的做法是这样：把查询转成嵌入（转换成能表达含义的数字），对所有已存记忆计算余弦相似度，返回前 K 条结果。这在简单场景可行，但在实践中会失灵。

纯语义检索会漏掉技术领域中重要的精确关键词匹配。当多条记忆内容相似时，它会返回重复的结果。当候选池很大时，它也无法区分高度相关和仅仅沾边的结果。

**记忆检索模式**就是解决这些问题的策略和算法。它们来自几十年的信息检索研究。在本 notebook 中，你将从零构建一个多阶段检索流程，组合语义检索、BM25 关键词匹配、分数融合、重排序和多样性过滤。

**学完本 notebook，你将理解：**
- 如何组合嵌入检索和关键词检索来提升查全率。
- 倒数排名融合（Reciprocal Rank Fusion）如何合并来自不同来源的排名列表。
- 最大边际相关性（MMR，Maximal Marginal Relevance）如何避免重复结果。
- 假设文档嵌入（HyDE，Hypothetical Document Embeddings）如何改进查询匹配。
- 每种模式何时有效、何时适得其反。

## 核心概念

- **嵌入（embedding）**：一串数字（即向量），表示一段文本的含义。含义相近的文本产生相近的向量。我们用嵌入模型把文本转换成这些向量。

- **余弦相似度（cosine similarity）**：衡量两个向量相似程度的指标。取值范围从 -1（方向相反）到 1（方向相同）。对归一化的嵌入来说，余弦相似度等于点积。

- **BM25**：来自信息检索领域的关键词匹配算法，全称 "Best Matching 25"。它根据词频（一个词出现的次数）、逆文档频率（这个词在所有文档中有多罕见）和文档长度给文档打分。

- **混合检索（hybrid retrieval）**：同时运行语义检索（基于嵌入）和词法检索（基于关键词），再合并两者结果。这样既能命中语义相关的结果，也能命中精确关键词的结果。

- **倒数排名融合（Reciprocal Rank Fusion，RRF）**：把多个排名列表合并成一个的方法。对每个文档，RRF 在所有列表上累加 `score = sum(1 / (k + rank))`。在多个来源中都排名靠前的文档会得到最高的综合分数。

- **最大边际相关性（Maximal Marginal Relevance，MMR）**：一种在相关性和多样性之间取得平衡的结果选择算法。每一步它都选出与查询最相关、但与已选结果最不相似的候选。公式为 `MMR = lambda * sim(doc, query) - (1 - lambda) * max(sim(doc, selected))`。

- **交叉编码器重排序（cross-encoder re-ranking）**：一种第二阶段模型，输入 (查询, 文档) 对，输出一个精确的相关性分数。比单独用嵌入相似度准确得多，但速度太慢，不适合搜索大集合。所以只把它用在少量候选上。

- **假设文档嵌入（HyDE，Hypothetical Document Embeddings）**：一种查询转换技术。先让 LLM 为查询生成一个假设性答案，再用这个答案做嵌入去搜索相似文档。这弥补了短查询和较长的已存记忆之间的词汇差异。

- **token**：LLM 内部处理的词片段。英文中约 4 个字符对应 1 个 token。上下文窗口以 token 计量。

## 架构

检索流程像一个多阶段漏斗。每一阶段都收窄候选集，同时提升质量。

<p align="center">
  <img src="../../images/diagrams/20_memory_retrieval_patterns.svg" alt="记忆检索模式架构图" width="720"/>
</p>

**数据流：** 查询从左侧进入。一个可选的查询转换器（HyDE 或查询扩展）先改写它。转换后的查询并行进入两个索引：语义索引（基于嵌入的搜索）和 BM25 索引（关键词搜索）。分数融合（倒数排名融合）把两份排名列表合并成一份。元数据过滤器按时间范围、记忆类型或来源收窄候选。重排序器（交叉编码器）以更高的精度对剩余候选重新打分。最后，MMR 在最大化多样性的同时选出前 K 条结果。这些结果进入智能体的上下文窗口。

## 环境准备

安装依赖。我们用 `openai` 做嵌入和对话，用 `rank_bm25` 做词法检索，用 `numpy` 做向量运算，用 `sentence-transformers` 做交叉编码器重排序。

In [ ]:
%pip install -q python-dotenv rank_bm25 numpy sentence-transformers deepagents langchain-openai

导入库并配置 API 客户端。你需要在 `.env` 文件里配置 `LLM_API_KEY`。

In [ ]:
import os
import json
from dataclasses import dataclass, field
from typing import Optional
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)
def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


网关没有 `/embeddings` 接口时，用这份本地确定性向量兜底：词加字符 n-gram 哈希后归一化。配置 `LLM_EMBEDDING_BASE_URL` 和 `LLM_EMBEDDING_MODEL_NAME` 后，自动切换到真实向量端点。


In [ ]:
EMBEDDING_MODEL = os.getenv("LLM_EMBEDDING_MODEL_NAME", "")


def embed_texts(texts: list[str]) -> list[list[float]]:
    """把文本转成向量。

    当前网关没有 /embeddings 接口。默认退回到本地确定性向量：
    词 + 字符 n-gram 哈希后归一化，语义相近的文本仍会得到相近向量，
    足够演示记忆检索的几何逻辑。配置 LLM_EMBEDDING_BASE_URL 和
    LLM_EMBEDDING_MODEL_NAME 后，自动改用真实向量端点。
    """
    if EMBEDDING_MODEL:
        from openai import OpenAI

        client = OpenAI(
            api_key=os.getenv("LLM_EMBEDDING_API_KEY", os.getenv("LLM_API_KEY")),
            base_url=os.getenv("LLM_EMBEDDING_BASE_URL", os.getenv("LLM_BASE_URL")),
        )
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
        return [item.embedding for item in response.data]

    import hashlib
    import math
    import re as re_mod

    dim = 512
    vectors = []
    for text in texts:
        vec = [0.0] * dim
        for word in re_mod.findall(r"\w+", text.lower()):
            grams = {word} | {word[i:i + 3] for i in range(max(1, len(word) - 2))}
            for gram in grams:
                digest = hashlib.blake2b(gram.encode("utf-8"), digest_size=8).digest()
                index = int.from_bytes(digest, "big") % dim
                vec[index] += 1.0
        norm = math.sqrt(sum(v * v for v in vec)) or 1.0
        vectors.append([v / norm for v in vec])
    return vectors


## 实现

我们将逐步构建检索流程。每个部件都是独立组件。最后，我们把它们组装成一个 `HybridRetriever` 类。

### 第 1 步：记忆存储

首先，我们需要一个表示记忆的数据结构，以及一个保存记忆的存储。

In [ ]:
@dataclass
class Memory:
    """单条记忆，包含文本、嵌入和元数据。"""
    text: str
    memory_id: str = ""
    embedding: Optional[np.ndarray] = field(default=None, repr=False)
    metadata: dict = field(default_factory=dict)
    timestamp: float = 0.0

    def __post_init__(self):
        if not self.memory_id:
            # 根据内容生成确定性 ID
            self.memory_id = hashlib.sha256(self.text.encode()).hexdigest()[:12]


class MemoryStore:
    """保存记忆及其嵌入的内存存储。"""

    def __init__(self):
        self.memories: list[Memory] = []

    def add(self, text: str, metadata: dict | None = None, timestamp: float = 0.0) -> Memory:
        mem = Memory(text=text, metadata=metadata or {}, timestamp=timestamp)
        self.memories.append(mem)
        return mem

    def compute_embeddings(self, embed_fn):
        """为所有缺少嵌入的记忆计算嵌入。"""
        texts_to_embed = []
        indices = []
        for i, mem in enumerate(self.memories):
            if mem.embedding is None:
                texts_to_embed.append(mem.text)
                indices.append(i)

        if texts_to_embed:
            embeddings = embed_fn(texts_to_embed)
            for idx, emb in zip(indices, embeddings):
                self.memories[idx].embedding = emb

    def __len__(self) -> int:
        return len(self.memories)

    def __repr__(self) -> str:
        return f"MemoryStore({len(self.memories)} memories)"

### 第 2 步：嵌入辅助函数

我们需要一个函数，用 OpenAI API 把文本转换成嵌入。这个函数支持批量处理以提高效率。

In [ ]:
def get_embeddings(texts: list[str]) -> list[np.ndarray]:
    """用 OpenAI API 获取一组文本的嵌入。"""
    return [np.array(v, dtype=np.float32) for v in embed_texts(texts)]


def get_embedding(text: str, model: str = EMBEDDING_MODEL) -> np.ndarray:
    """获取单个嵌入。"""
    return get_embeddings([text], model=model)[0]

### 第 3 步：语义检索

想象一家书店，在地图上按主题摆放图书。相似的书放在一起。要找相关的书，你先在地图上定位查询的位置，然后拿起最近的那几本。基于嵌入的语义检索就是这个原理。

余弦相似度衡量两个向量之间的夹角。值为 1.0 表示方向完全相同（含义相同）。值为 0.0 表示没有关系。

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """计算两个向量之间的余弦相似度。"""
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))


def semantic_search(
    query_embedding: np.ndarray,
    memories: list[Memory],
    top_k: int = 10,
) -> list[tuple[Memory, float]]:
    """按与查询嵌入的余弦相似度给记忆排序。

    返回 (记忆, 分数) 元组列表，按分数降序排列。
    """
    scored = []
    for mem in memories:
        if mem.embedding is not None:
            score = cosine_similarity(query_embedding, mem.embedding)
            scored.append((mem, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

### 第 4 步：BM25 词法检索

想象老图书馆里的卡片目录。你查一个精确的书名或关键词，目录告诉你书在哪个书架。BM25 就是这样工作的。它根据文档的词与查询词的匹配程度给文档打分。

嵌入不擅长的地方，BM25 恰好擅长：罕见的技术术语、缩写词和精确短语。`rank_bm25` 库提供了现成的实现。

In [ ]:
class BM25Index:
    """封装 rank_bm25，对记忆做词法检索。"""

    def __init__(self, memories: list[Memory]):
        self.memories = memories
        # 按空白切分并转小写来分词
        tokenized = [mem.text.lower().split() for mem in memories]
        self.bm25 = BM25Okapi(tokenized)

    def search(self, query: str, top_k: int = 10) -> list[tuple[Memory, float]]:
        """用 BM25 打分进行检索。返回 (记忆, 分数) 对。"""
        query_tokens = query.lower().split()
        scores = self.bm25.get_scores(query_tokens)

        # 把记忆和分数配对并降序排序
        scored = [(self.memories[i], float(scores[i])) for i in range(len(self.memories))]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]

### 第 5 步：倒数排名融合（RRF）

你向两个朋友请教餐厅推荐。朋友 A 给你一份排名列表，朋友 B 给你另一份。有些餐厅两份列表里都有，有些只出现在其中一份。你怎么把它们合并起来？

RRF 解决了这个问题。对每个条目，它在所有列表上累加 `1 / (k + rank)`。被多个来源排在前面的条目会得到最高的综合分数。常数 `k`（通常取 60）用于防止排名最靠前的条目权重过大。

In [ ]:
def reciprocal_rank_fusion(
    *ranked_lists: list[tuple[Memory, float]],
    k: int = 60,
) -> list[tuple[Memory, float]]:
    """用倒数排名融合合并多个排名列表。

    Args:
        ranked_lists: 每个列表包含按排名顺序排列的 (记忆, 分数) 元组。
        k: 平滑常数。取值越大，排名位置的影响越小。

    Returns:
        融合后的 (记忆, rrf_score) 列表，按降序排列。
    """
    rrf_scores: dict[str, float] = {}
    memory_lookup: dict[str, Memory] = {}

    for ranked_list in ranked_lists:
        for rank, (mem, _score) in enumerate(ranked_list):
            rrf_scores[mem.memory_id] = rrf_scores.get(mem.memory_id, 0.0) + 1.0 / (k + rank + 1)
            memory_lookup[mem.memory_id] = mem

    # 按融合分数降序排序
    fused = [(memory_lookup[mid], score) for mid, score in rrf_scores.items()]
    fused.sort(key=lambda x: x[1], reverse=True)
    return fused

### 第 6 步：交叉编码器重排序

想象面试的两个阶段。第一阶段（简历筛选）快但粗糙。第二阶段（现场面试）慢但准确。重排序也是同样的道理。

第一阶段（语义检索或 BM25）快，因为它比较的是预先算好的表示。交叉编码器重排序器慢，因为它要把查询和文档放在一起逐词阅读。但这种联合阅读能产生准确得多的相关性分数。

我们用 `sentence-transformers` 库加载交叉编码器模型。它对每个 (查询, 文档) 对给出一个相关性分数。

In [ ]:
class Reranker:
    """基于 sentence-transformers 的交叉编码器重排序器。"""

    def __init__(self, model_name: str = CROSS_ENCODER_MODEL):
        self.model = CrossEncoder(model_name)

    def rerank(
        self,
        query: str,
        candidates: list[tuple[Memory, float]],
        top_k: int | None = None,
    ) -> list[tuple[Memory, float]]:
        """用交叉编码器给候选重新打分。

        Args:
            query: 搜索查询。
            candidates: 前面阶段输出的 (记忆, 初始分数) 列表。
            top_k: 只返回这么多条结果。None 表示全部返回。

        Returns:
            重排序后的 (记忆, cross_encoder_score) 列表。
        """
        if not candidates:
            return []

        # 构建交叉编码器的输入对
        pairs = [(query, mem.text) for mem, _ in candidates]
        scores = self.model.predict(pairs)

        # 把记忆和新分数配对
        reranked = [(candidates[i][0], float(scores[i])) for i in range(len(candidates))]
        reranked.sort(key=lambda x: x[1], reverse=True)

        if top_k is not None:
            reranked = reranked[:top_k]
        return reranked

### 第 7 步：最大边际相关性（MMR）

想象你在创建播放列表。你不会想连续听五首悲伤的情歌，哪怕它们都很动听。你想要多样性。MMR 把同样的思路用在搜索结果上。

每一步，MMR 都选出与查询最相关、但与已选结果最不相似的候选。`lambda_param` 控制两者的平衡。取 1.0 表示「只选最相关的，不管多样性」。取 0.0 表示「只选与已有结果最不同的」。实践中取 0.5 到 0.7 之间效果较好。

贪心选择循环是 MMR 的核心。它每次选出一个结果，每次选择都最大化 `lambda * relevance - (1 - lambda) * similarity_to_already_picked`。我们把它抽成一个辅助函数，让两个函数都保持简短易读。

In [ ]:
def _mmr_greedy_select(
    candidates: list[tuple[Memory, float]],
    query_embedding: np.ndarray,
    top_k: int,
    min_score: float,
    score_range: float,
    lambda_param: float,
) -> list[tuple[Memory, float]]:
    """MMR 的贪心选择循环。

    每一步选出 MMR 分数最高的候选：
    lambda * relevance - (1 - lambda) * max_similarity_to_selected。
    """
    remaining = list(range(len(candidates)))
    selected_embeddings: list[np.ndarray] = []
    result = []

    for _ in range(min(top_k, len(candidates))):
        best_idx = -1
        best_mmr = -float("inf")

        for idx in remaining:
            mem, raw_score = candidates[idx]
            if mem.embedding is None:
                continue

            # 相关性分量：归一化后的分数
            relevance = (raw_score - min_score) / score_range

            # 多样性分量：与任一已选结果的最大相似度
            if selected_embeddings:
                similarities = [
                    cosine_similarity(mem.embedding, sel_emb)
                    for sel_emb in selected_embeddings
                ]
                max_sim = max(similarities)
            else:
                max_sim = 0.0

            mmr_score = lambda_param * relevance - (1 - lambda_param) * max_sim

            if mmr_score > best_mmr:
                best_mmr = mmr_score
                best_idx = idx

        if best_idx == -1:
            break

        mem, score = candidates[best_idx]
        result.append((mem, best_mmr))
        selected_embeddings.append(mem.embedding)
        remaining.remove(best_idx)

    return result


主函数 `mmr_selection` 先把分数归一化到 0 到 1 区间，再把剩下的工作交给贪心循环。调用方只需要和这个函数打交道。

In [ ]:
def mmr_selection(
    query_embedding: np.ndarray,
    candidates: list[tuple[Memory, float]],
    top_k: int = 5,
    lambda_param: float = 0.7,
) -> list[tuple[Memory, float]]:
    """用最大边际相关性选择结果。

    在与查询的相关性和已选结果之间的多样性之间取得平衡。

    Args:
        query_embedding: 查询的嵌入向量。
        candidates: 预先打好分的 (记忆, relevance_score) 对。
        top_k: 要选择的结果数量。
        lambda_param: 相关性（1.0）与多样性（0.0）之间的平衡。

    Returns:
        选出的 (记忆, mmr_score) 对。
    """
    if not candidates:
        return []

    # 把相关性分数归一化到 [0, 1]
    scores = [s for _, s in candidates]
    max_score = max(scores) if scores else 1.0
    min_score = min(scores) if scores else 0.0
    score_range = max_score - min_score if max_score != min_score else 1.0

    return _mmr_greedy_select(
        candidates, query_embedding, top_k,
        min_score, score_range, lambda_param,
    )


### 第 8 步：HyDE 查询转换

想象在两种语言之间翻译。你的查询说的是「问题语言」，而你的文档说的是「答案语言」。HyDE 让 LLM 写一个假设性答案来弥合这个差异，然后用这个答案而不是原问题去搜索。

这个方法有效，是因为假设性答案和真实存储的记忆共享词汇和结构。即使假设性答案是错的，它的嵌入也会比原始短查询的嵌入更接近相关文档。

In [ ]:
def hyde_transform(query: str) -> str:
    """用 LLM 为查询生成一个假设性文档（答案）。

    之后用这个假设性答案（而不是原始查询）做嵌入来检索。
    这弥合了问题和文档之间的词汇差异。
    """
    response = llm.invoke([
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Given a question, write a short "
                    "paragraph (3-4 sentences) that would be a good answer. "
                    "Write as if this answer exists in a knowledge base. "
                    "State facts directly. Do not hedge."
                ),
            },
            {"role": "user", "content": query},
        ])
    return to_text(response.content)

### 第 9 步：完整的混合检索器

现在我们把所有组件组装成一个类。`retrieve` 方法运行完整流程：可选的 HyDE 转换、并行的语义检索和 BM25 检索、RRF 融合、交叉编码器重排序，以及 MMR 多样性选择。

In [ ]:
class HybridRetriever:
    """结合语义检索和词法检索的多阶段检索流程。

    流程阶段：
    1. 查询转换（可选 HyDE）
    2. 并行的语义检索 + BM25 检索
    3. 倒数排名融合
    4. 交叉编码器重排序
    5. MMR 多样性选择
    """

    def __init__(self, store: MemoryStore, use_reranker: bool = True):
        self.store = store
        self.bm25_index: BM25Index | None = None
        self.reranker = Reranker() if use_reranker else None

    def build_indices(self):
        """计算嵌入并构建 BM25 索引。"""
        self.store.compute_embeddings(get_embeddings)
        self.bm25_index = BM25Index(self.store.memories)
        print(f"Built indices over {len(self.store)} memories.")


`retrieve` 方法运行完整流程。它把所有阶段串在一起：可选的 HyDE 转换、并行的语义检索和 BM25 检索、RRF 融合、交叉编码器重排序，以及 MMR 多样性选择。我们把它单独挂到类上，让每个单元格只专注一件事。

`_retrieve_candidates` 负责流程的前三个阶段：可选的 HyDE 转换、并行的语义检索和 BM25 检索，以及倒数排名融合。它返回融合后的候选列表和查询嵌入。

In [ ]:
def _retrieve_candidates(
    self,
    query: str,
    initial_k: int,
    use_hyde: bool,
) -> tuple[list[tuple[Memory, float]], "np.ndarray", str]:
    """阶段 1-3：查询转换、并行检索和 RRF 融合。

    Returns:
        (fused_results, query_embedding, search_text) 元组。
    """
    # 阶段 1：查询转换
    search_text = query
    if use_hyde:
        hypothetical_answer = hyde_transform(query)
        search_text = hypothetical_answer

    # 阶段 2：并行检索
    query_embedding = get_embedding(search_text)
    semantic_results = semantic_search(
        query_embedding, self.store.memories, top_k=initial_k
    )

    # BM25 始终使用原始查询（关键词很重要）
    if self.bm25_index:
        bm25_results = self.bm25_index.search(query, top_k=initial_k)
    else:
        bm25_results = []

    # 阶段 3：倒数排名融合
    if bm25_results:
        fused = reciprocal_rank_fusion(semantic_results, bm25_results, k=60)
    else:
        fused = semantic_results

    return fused, query_embedding, search_text

HybridRetriever._retrieve_candidates = _retrieve_candidates


`retrieve` 方法编排整个流程。它调用 `_retrieve_candidates` 完成阶段 1-3，然后应用交叉编码器重排序（阶段 4）和 MMR 多样性选择（阶段 5）。

In [ ]:
def retrieve(
    self,
    query: str,
    top_k: int = 5,
    use_hyde: bool = False,
    use_reranker: bool = True,
    use_mmr: bool = True,
    lambda_param: float = 0.7,
    initial_k_multiplier: int = 4,
) -> list[tuple[Memory, float]]:
    """运行完整的检索流程。

    Args:
        query: 搜索查询。
        top_k: 最终结果的数量。
        use_hyde: 是否应用 HyDE 查询转换。
        use_reranker: 是否应用交叉编码器重排序。
        use_mmr: 是否应用 MMR 多样性过滤。
        lambda_param: MMR 的相关性与多样性平衡（0.0 到 1.0）。
        initial_k_multiplier: 第一阶段获取多少候选（top_k 的倍数）。

    Returns:
        (记忆, 分数) 元组列表。
    """
    initial_k = top_k * initial_k_multiplier

    # 阶段 1-3：通过语义检索 + BM25 + 融合得到候选
    fused, query_embedding, search_text = self._retrieve_candidates(
        query, initial_k, use_hyde,
    )

    # 阶段 4：交叉编码器重排序
    rerank_candidates = fused[: top_k * 3]
    if use_reranker and self.reranker and rerank_candidates:
        reranked = self.reranker.rerank(
            query, rerank_candidates, top_k=top_k * 2
        )
    else:
        reranked = rerank_candidates

    # 阶段 5：MMR 多样性选择
    if use_mmr:
        if use_hyde:
            original_query_embedding = get_embedding(query)
        else:
            original_query_embedding = query_embedding
        final = mmr_selection(
            original_query_embedding, reranked,
            top_k=top_k, lambda_param=lambda_param,
        )
    else:
        final = reranked[:top_k]

    return final

HybridRetriever.retrieve = retrieve


## 示例运行

我们用一组贴近真实的记忆集合来测试检索流程。先向存储中填充关于一个软件项目的记忆，然后运行一组查询，检验检索在不同方面的表现。

In [ ]:
# 创建一个记忆存储，放入多样的项目记忆
store = MemoryStore()

project_memories = [
    # 技术决策
    "We chose PostgreSQL as our primary database because it handles JSON well and supports full-text search.",
    "The team adopted FastAPI for the backend REST API. It has built-in OpenAPI docs and async support.",
    "We use Redis for caching API responses. Cache TTL is set to 300 seconds for most endpoints.",
    "Authentication uses JWT tokens with a 24-hour expiry. Refresh tokens last 30 days.",
    "The frontend is built with React 18 and TypeScript. We use Zustand for state management.",

    # 团队与流程
    "Sprint retrospective on March 15: the team agreed to write integration tests before merging PRs.",
    "Alice leads the backend team. Bob handles DevOps. Carol manages the frontend.",
    "We deploy to AWS using Terraform. The CI/CD pipeline runs on GitHub Actions.",
    "Code reviews require two approvals before merging to the main branch.",
    "The on-call rotation is weekly: Alice, Bob, Carol, then Dave.",

    # 缺陷报告与事故
    "Bug: the /users endpoint returns a 500 error when the email field contains unicode characters.",
    "Incident on April 2: Redis cache went down for 40 minutes. Fallback to direct DB queries worked.",
    "Memory leak in the WebSocket handler was traced to unclosed connections in the event loop.",
    "Performance issue: the search endpoint takes 3 seconds for queries over 100 characters.",

    # 架构笔记
    "The message queue uses RabbitMQ. Each microservice has its own exchange and dead-letter queue.",
    "Database migrations run automatically on deployment using Alembic.",
    "Logs are shipped to Elasticsearch via Fluentd. Kibana dashboards show error rates and latency.",
    "The recommendation engine uses cosine similarity over user-item embeddings stored in Pinecone.",

    # 会议记录
    "Product meeting April 10: we decided to prioritize the search feature over the notification system.",
    "Architecture review: proposed moving from REST to gRPC for internal service communication.",
]

for i, text in enumerate(project_memories):
    store.add(text, metadata={"source": "project_log", "index": i}, timestamp=float(i))

print(f"Added {len(store)} memories to the store.")

构建索引。这一步为所有记忆计算嵌入，并创建 BM25 索引。

In [ ]:
retriever = HybridRetriever(store, use_reranker=True)
retriever.build_indices()

### 查询 1：语义型查询

我们来搜索一个需要理解含义、而不是精确关键词匹配的问题。

In [ ]:
query = "What database technology do we use and why?"
results = retriever.retrieve(query, top_k=3, use_hyde=False, use_reranker=True, use_mmr=True)

print(f"Query: '{query}'\n")
for i, (mem, score) in enumerate(results):
    print(f"  {i+1}. [score={score:.4f}] {mem.text}")
    print()

### 查询 2：关键词密集型查询

这个查询使用精确术语。这时 BM25 的贡献应该比语义检索更大。

In [ ]:
query = "Redis cache TTL"
results = retriever.retrieve(query, top_k=3, use_hyde=False, use_reranker=True, use_mmr=True)

print(f"Query: '{query}'\n")
for i, (mem, score) in enumerate(results):
    print(f"  {i+1}. [score={score:.4f}] {mem.text}")
    print()

### 对比不同的检索策略

我们用不同的流程配置运行同一个查询，看看每个组件各贡献了什么。

In [ ]:
query = "What went wrong in production recently?"

configurations = [
    ("Semantic only", dict(use_hyde=False, use_reranker=False, use_mmr=False)),
    ("BM25 + Semantic (RRF)", dict(use_hyde=False, use_reranker=False, use_mmr=False)),
    ("+ Re-ranking", dict(use_hyde=False, use_reranker=True, use_mmr=False)),
    ("+ MMR diversity", dict(use_hyde=False, use_reranker=True, use_mmr=True, lambda_param=0.5)),
    ("+ HyDE", dict(use_hyde=True, use_reranker=True, use_mmr=True, lambda_param=0.5)),
]

print(f"Query: '{query}'\n")
print("=" * 90)

for name, config in configurations:
    results = retriever.retrieve(query, top_k=3, **config)
    print(f"\n[{name}]")
    for i, (mem, score) in enumerate(results):
        preview = mem.text[:90] + ("..." if len(mem.text) > 90 else "")
        print(f"  {i+1}. [{score:.4f}] {preview}")

### MMR 中 lambda 的影响

看看 `lambda_param` 如何改变结果。lambda 越低，多样性越强；lambda 越高，越偏向相关性。

In [ ]:
query = "Tell me about our infrastructure and deployment"

lambdas = [0.3, 0.5, 0.7, 0.9]

print(f"Query: '{query}'\n")

for lam in lambdas:
    results = retriever.retrieve(
        query, top_k=4, use_reranker=True, use_mmr=True, lambda_param=lam
    )
    print(f"lambda={lam}:")
    for i, (mem, score) in enumerate(results):
        preview = mem.text[:85] + ("..." if len(mem.text) > 85 else "")
        print(f"  {i+1}. {preview}")
    print()

### 在对话中使用检索到的记忆

下面演示如何把检索到的记忆送进 LLM 调用。检索器负责找到相关上下文，LLM 用它来回答。

In [ ]:
def answer_with_memory(query: str, retriever: HybridRetriever, top_k: int = 5) -> str:
    """检索相关记忆，并用它们回答查询。"""
    results = retriever.retrieve(
        query, top_k=top_k, use_hyde=False, use_reranker=True, use_mmr=True
    )

    # 把记忆格式化为上下文
    context_parts = []
    for i, (mem, score) in enumerate(results):
        context_parts.append(f"[Memory {i+1}] {mem.text}")
    context = "\n".join(context_parts)

    response = llm.invoke([
            {
                "role": "system",
                "content": (
                    "You are a helpful project assistant. Answer the user's question "
                    "based on the retrieved memories below. If the memories do not contain "
                    "enough information, say so. Keep your answer concise (2-4 sentences).\n\n"
                    f"Retrieved memories:\n{context}"
                ),
            },
            {"role": "user", "content": query},
        ])
    return to_text(response.content)


# 用几个问题测试
questions = [
    "Who should I talk to about the CI/CD pipeline?",
    "What happened with the Redis incident?",
    "How does our search feature work?",
]

for q in questions:
    print(f"Q: {q}")
    answer = answer_with_memory(q, retriever)
    print(f"A: {answer}\n")

## 权衡

### 这些模式何时有效

- **混合检索能接住任何单一方法漏掉的东西。** 语义检索处理转述和含义，BM25 处理精确术语、缩写词和罕见词。两者结合后，各类查询的查全率都会提升。
- **MMR 防止结果重复。** 没有多样性过滤时，你常常会拿到同一条记忆的五个变体。MMR 确保智能体看到存储知识更全面的样子。
- **重排序提升头部结果的查准率。** 交叉编码器把查询和文档放在一起阅读，能捕捉嵌入相似度遗漏的细微相关性信号。
- **HyDE 帮助处理简短或模糊的查询。** 当查询和存储记忆没有共同词汇时，一个假设性答案可以弥合差距。

### 这些模式何时有害

- **每个流程阶段都会增加延迟。** 嵌入查询只需几毫秒，BM25 也很快。但交叉编码器重排序器按候选数量不同会增加 50 到 200 毫秒，HyDE 还要多一次完整的 LLM 调用（200 到 1000 毫秒）。实时聊天可能需要跳过某些阶段。
- **复杂度上升。** 每个组件都有要调优的参数：RRF 的 `k`、MMR 的 `lambda`、传给每个阶段的候选数量。活动部件越多，配置出错的机会越大。
- **小型记忆库收益有限。** 如果记忆少于 50 条，单独一次余弦相似度语义检索就够用了。多阶段流程要在候选池很大时（数百到数千条记忆）才能发挥优势。
- **HyDE 在事实型查询上可能误导。** 如果 LLM 生成了错误的假设性答案，嵌入搜索会找到与错误答案相似的文档。HyDE 最适合开放式查询，不适合精确的事实查找。

## 延伸阅读

- Robertson, S., & Zaragoza, H. (2009). ["The Probabilistic Relevance Framework: BM25 and Beyond."](https://doi.org/10.1561/1500000019) *Foundations and Trends in Information Retrieval*, 3(4), 333-389。BM25 的权威参考文献。

- Carbonell, J., & Goldstein, J. (1998). ["The Use of MMR, Diversity-Based Reranking for Reordering Documents and Producing Summaries."](https://doi.org/10.1145/290941.291025) *ACM SIGIR*, 335-336。MMR 的原始论文。

- Gao, L., et al. (2022). ["Precise Zero-Shot Dense Retrieval without Relevance Labels (HyDE)."](https://arxiv.org/abs/2212.10496) 表明先生成假设性答案再对其做嵌入，可以提升零样本检索的效果。

- Ma, X., et al. (2023). ["Fine-Tuning LLaMA for Multi-Stage Text Retrieval."](https://arxiv.org/abs/2310.08319) 探索在多阶段流程中把 LLM 用作检索器和重排序器。

- [Sentence Transformers: Cross-Encoders](https://www.sbert.net/docs/cross_encoder/usage/usage.html) 重排序阶段所用交叉编码器模型的文档。

- [rank-bm25 on PyPI](https://pypi.org/project/rank-bm25/) 本 notebook 用于词法检索的 BM25 库。

---

*上一篇：[19：遗忘与衰减](../19_forgetting_and_decay/) | 下一篇：[21：跨会话记忆](../21_cross_session_memory/)*

## 🧪 动手练习

三个小挑战，加深你的理解。每个大约需要 10 到 30 分钟。

### 挑战 1：融合权重调优
在 `reciprocal_rank_fusion()` 中调整语义结果与 BM25 结果的相对权重。试试 0.3/0.7、0.5/0.5 和 0.7/0.3 三种比例。每种设置运行 10 个查询并测量 MRR（平均倒数排名）。找出对你的数据效果最好的比例。

### 挑战 2：重排序器的影响
测量应用 `Reranker` 前后的 Recall@10 和 Precision@10。运行 20 个查询，记录重排序后头部结果的位置变化，并计算重排序把更好的结果推到第 1 位的频率。

### 挑战 3：用 HyDE 做记忆检索
对 10 个记忆查询应用 `hyde_transform()`，用 `semantic_search()` 比较原始查询与 HyDE 转换后查询的前 5 条结果。统计 HyDE 方法多找出了多少条相关记忆。用技术 06 向量库记忆中的向量库来做这个测试。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--20-memory-retrieval-patterns--memory-retrieval-patterns)